In [6]:
!pip install pandas scikit-learn xgboost joblib seaborn

**Import Libraries**

In [7]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

Load Data Sets

In [8]:
def load_datasets():
    
    symptom_df = pd.read_csv("data.csv")

    clinical_df = pd.read_csv("clinic.csv")

    env_df = pd.read_csv("env.csv")

    print("Datasets loaded successfully")

    return symptom_df, clinical_df, env_df

Preprocessing Function

In [9]:
def preprocess_data(symptom_df, clinical_df, env_df):
    
    symptom_df = symptom_df.fillna(0)

    clinical_df = clinical_df.fillna(0)

    env_df = env_df.fillna(0)

    print("Missing values handled")

    return symptom_df, clinical_df, env_df

Encode Disease Labels

In [10]:
def encode_labels(symptom_df):
    
    encoder = LabelEncoder()

    symptom_df["Disease"] = encoder.fit_transform(
        symptom_df["Disease"]
    )

    joblib.dump(
        encoder,
        "disease_encoder.pkl"
    )

    print("Disease labels encoded")

    return symptom_df, encoder

Split Dataset Function

In [11]:
def split_data(X, y):
    
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    return X_train, X_test, y_train, y_test

Model Evaluation Function

In [12]:
def evaluate_model(y_test, predictions):
    
    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted"
    )

    print("Accuracy:", accuracy)

    print("Precision:", precision)

    print("Recall:", recall)

    print("F1 Score:", f1)

    print("\nClassification Report")

    print(
        classification_report(
            y_test,
            predictions
        )
    )

    return accuracy, precision, recall, f1

Confusion Matrix Function

In [13]:
def plot_confusion_matrix(y_test, predictions):
    
    cm = confusion_matrix(
        y_test,
        predictions
    )

    plt.figure()

    sns.heatmap(
        cm,
        annot=True,
        fmt="d"
    )

    plt.title("Confusion Matrix")

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.show()

Train Symptom Model (XGBoost)

In [14]:
def train_symptom_model(symptom_df):
    
    X = symptom_df.drop(
        "Disease",
        axis=1
    )

    y = symptom_df["Disease"]

    X_train, X_test, y_train, y_test = split_data(
        X,
        y
    )

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    evaluate_model(
        y_test,
        predictions
    )

    plot_confusion_matrix(
        y_test,
        predictions
    )

    joblib.dump(
        model,
        "xgb_model.pkl"
    )

    joblib.dump(
        X.columns.tolist(),
        "symptom_features.pkl"
    )

    joblib.dump(
        symptom_df,
        "symptom_data.pkl"
    )

    print("Symptom model saved")

    return model

Train Clinical Model

In [15]:
def train_clinical_model(clinical_df):
    
    X = clinical_df.drop(
        "Disease",
        axis=1
    )

    y = clinical_df["Disease"]

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X
    )

    X_train, X_test, y_train, y_test = split_data(
        X_scaled,
        y
    )

    model = LogisticRegression(
        max_iter=1000
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    evaluate_model(
        y_test,
        predictions
    )

    plot_confusion_matrix(
        y_test,
        predictions
    )

    joblib.dump(
        model,
        "clinical_model.pkl"
    )

    joblib.dump(
        scaler,
        "clinical_scaler.pkl"
    )

    joblib.dump(
        X.columns.tolist(),
        "clinical_features.pkl"
    )

    print("Clinical model saved")

    return model

Train Environmental Model

In [16]:
def train_env_model(env_df):
    
    X = env_df.drop(
        "Disease",
        axis=1
    )

    y = env_df["Disease"]

    X_train, X_test, y_train, y_test = split_data(
        X,
        y
    )

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    evaluate_model(
        y_test,
        predictions
    )

    plot_confusion_matrix(
        y_test,
        predictions
    )

    joblib.dump(
        model,
        "env_model.pkl"
    )

    joblib.dump(
        X.columns.tolist(),
        "env_features.pkl"
    )

    print("Environmental model saved")

    return model

Ensemble Prediction Function

In [17]:
def ensemble_prediction(
    symptom_model,
    clinical_model,
    env_model,
    symptom_input,
    clinical_input,
    env_input
):

    prob_symptom = symptom_model.predict_proba(
        symptom_input
    )

    prob_clinical = clinical_model.predict_proba(
        clinical_input
    )

    prob_env = env_model.predict_proba(
        env_input
    )

    final_probability = (

        prob_env * 0.65

        + prob_clinical * 0.18

        + prob_symptom * 0.17

    )

    return final_probability

In [18]:
def run_training_pipeline():
    
    symptom_df, clinical_df, env_df = load_datasets()

    symptom_df, clinical_df, env_df = preprocess_data(
        symptom_df,
        clinical_df,
        env_df
    )

    symptom_df, encoder = encode_labels(
        symptom_df
    )

    symptom_model = train_symptom_model(
        symptom_df
    )

    clinical_model = train_clinical_model(
        clinical_df
    )

    env_model = train_env_model(
        env_df
    )

    print("All models trained successfully")

In [19]:
run_training_pipeline()